# 06 -- Intensity measures (Arias, CAV, Housner, peaks)

In [1]:
import sys
sys.path.insert(0, "../src")            # run from the examples/ folder

import numpy as np
from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

# The example data folder (edit to your path if needed).
DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"

# A short window keeps every example fast (one ~10 min file).
WSTART, WLEN = "2026-05-31 15:00:00", "10min"

In [2]:
ds = SensorDataset(DATA, verbose=True)
ds.devices

------------------------------------------------------------
SensorDataset
------------------------------------------------------------
path        : C:\Users\ppala\Desktop\02_31MAY2026
files       : 32
time span   : 2026-05-31 14:52:12  ->  2026-05-31 20:02:13
duration    : 18601.0 s
devices     : MNAT0031, MNAT0034, MOF00134, MOF00135, MOF00136
fs / dt     : 252.5885 Hz / 0.003959 s
------------------------------------------------------------
axes (per sensor):
  MNAT0031   -> (3, 1, 5)
  MNAT0034   -> (3, 1, 5)
  MOF00134   -> (0, 1, 2)
  MOF00135   -> (3, 1, 5)
  MOF00136   -> (3, 1, 5)
------------------------------------------------------------
on-disk size: 782.17 MB
RAM         : used 33.07 GB / avail 0.44 GB (99%)
------------------------------------------------------------


['MNAT0031', 'MNAT0034', 'MOF00134', 'MOF00135', 'MOF00136']

In [3]:
wf = ds.MOF00135.window(WSTART, WLEN).baseline().filter(0.1, 24.9).derive()

## Arias

In [4]:
a = wf.arias(component="x", low=5, high=95)
print("t5-95:", round(float(a["t_start"]),2), "->", round(float(a["t_end"]),2), "s | IA_total:", round(float(a["IA_total"]),5))

C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


[arias] MOF00135 comp=x low=5 high=95 (computed)
t5-95: 31.01 -> 592.69 s | IA_total: 4e-05


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")


## CAV, Housner, peaks

In [5]:
print("CAV   =", round(float(wf.cav(component="x")["CAV"]),5))
print("SI    =", round(float(wf.housner(component="x", T1=0.1, T2=2.5)["SI"]),6))
print("peaks =", {ax: {k: round(float(v),5) for k,v in wf.peaks(component="all")[ax].items()} for ax in ["x","y","z"]})

[cav] MOF00135 comp=x (computed)
CAV   = 0.29345


[housner] MOF00135 comp=x T1=0.1 T2=2.5 (computed)
SI    = 0.001805
[peaks] MOF00135 comp=x PGA=0.005078 (computed)
[peaks] MOF00135 comp=y PGA=0.00257 (computed)
[peaks] MOF00135 comp=z PGA=0.008002 (computed)
[peaks] MOF00135 comp=x PGA=0.005078 (cached)
[peaks] MOF00135 comp=y PGA=0.00257 (cached)
[peaks] MOF00135 comp=z PGA=0.008002 (cached)
[peaks] MOF00135 comp=x PGA=0.005078 (cached)
[peaks] MOF00135 comp=y PGA=0.00257 (cached)
[peaks] MOF00135 comp=z PGA=0.008002 (cached)
peaks = {'x': {'PGA': 0.00508, 'PGV': 0.00167, 'PGD': 0.02353}, 'y': {'PGA': 0.00257, 'PGV': 0.00072, 'PGD': 0.03537}, 'z': {'PGA': 0.008, 'PGV': 0.00255, 'PGD': 0.09034}}


## Compare all sensors (broadcast)

In [6]:
allpk = ds.peaks(component="x")
print({d: round(float(allpk[d]["PGA"]),4) for d in sorted(allpk)})

C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MNAT0031' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


[peaks] MNAT0031 comp=x PGA=0.2672 (computed)


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MNAT0034' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


[peaks] MNAT0034 comp=x PGA=0.3852 (computed)


[peaks] MOF00134 comp=x PGA=0.133 (computed)


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


[peaks] MOF00135 comp=x PGA=0.6243 (computed)


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\examples\../src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00136' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


[peaks] MOF00136 comp=x PGA=0.6548 (computed)
{'MNAT0031': 0.2672, 'MNAT0034': 0.3852, 'MOF00134': 0.133, 'MOF00135': 0.6243, 'MOF00136': 0.6548}
